In [0]:
# create the widgests #

dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:

# read the data from bronze #

bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
# '''
# This code adds two timestamp columns:
# 1. insert_timestamp
# 2. update_timestamp
# '''


# from pyspark.sql import functions as F
# bronze_df.withColumn("insert_timestamp", F.current_timestamp())\
#          .withColumn("update_timestamp",F.current_timestamp())

In [0]:
# updated the records into silver table #

from pyspark.sql import functions as F
updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp())\
         .withColumn("update_timestamp",F.current_timestamp())

In [0]:
# 03/11/2026 - superseded by the load-type cell below (FULL / APPEND / MERGE)
# updated_df.write.mode("overwrite").saveAsTable(f"adb_caresync.silver.{table_name}")

In [0]:
# Update statement new query 03/11/2026#
from delta.tables import DeltaTable
from pyspark.sql import Window

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = (load_type or "FULL").strip().upper()   # blank -> FULL (previous behaviour)
keys = [k.strip() for k in merge_keys.split(",") if k.strip()]

# Default business keys per table - used when the caller (ADF) sends blank/invalid merge_keys (e.g. 'MERGE')
DEFAULT_MERGE_KEYS = {
    "patients": ["patient_id"],
    "hospitals": ["hospital_id"],
    "doctors": ["doctor_id"],
}
if load_type_norm == "MERGE" and (not keys or any(k not in updated_df.columns for k in keys)):
    fallback = DEFAULT_MERGE_KEYS.get(table_name.strip().lower())
    if fallback:
        print(f"WARNING: merge_keys '{merge_keys}' invalid for '{table_name}'; using default keys {fallback}")
        keys = fallback

if load_type_norm not in ("FULL", "APPEND", "MERGE"):
    raise ValueError(f"Invalid load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

target_exists = spark.catalog.tableExists(target_table)

if load_type_norm == "FULL":
    # Full load: replace target with the current batch
    updated_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

elif load_type_norm == "APPEND":
    # Append: add the current batch to the target
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    if not keys:
        raise ValueError("merge_keys is required for MERGE load (comma separated, e.g. 'hospital_id').")
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source: {missing}")

    # De-duplicate source on merge keys (MERGE fails if multiple source rows match one target row)
    if "updated_at" in updated_df.columns:
        w = Window.partitionBy(*keys).orderBy(F.col("updated_at").desc_nulls_last())
        source_df = updated_df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    else:
        source_df = updated_df.dropDuplicates(keys)

    if not target_exists:
        # First run: create the target table from the batch
        source_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        # On update keep the original insert_timestamp; refresh everything else incl. update_timestamp
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c not in keys and c != "insert_timestamp"}

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute())

print(f"{load_type_norm} load completed into {target_table}")


In [0]:
%sql
-- select * from adb_caresync.silver.hospitals
--# query to check data is landing or not #  

In [0]:
# record_count = updated_df.count()
# dbutils.notebook.exit(f"Record count: {record_count}")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))